In [1]:
from datasets import load_dataset
import torch
import torch.nn as nn
import numpy as np
import polars as pl
import io
from PIL import Image
from torchvision.transforms import v2

In [2]:
ds = load_dataset("dragonintelligence/CIFAKE-image-dataset")

In [3]:
df_train=ds['train'].to_polars()
df_test=ds['test'].to_polars()

### Converting the images from polars bytes into PIL images

In [4]:
def convert_images_into_bytes(t):
    d=t.with_columns(
        pl.col('image').struct.field('path').alias("image_path"),
        pl.col('image').struct.field('bytes').alias('image_bytes')
    )
    images=[Image.open(io.BytesIO(img)) for img in d['image_bytes'].to_list()]
    
    return images

train_images=convert_images_into_bytes(df_train)
test_images=convert_images_into_bytes(df_test)

### converting the images into pytorch suitable format --requires more memory

In [13]:
transform=v2.Compose([
      v2.ToImage(),
      v2.Resize(size=(224, 240)),
      v2.CenterCrop(224),
      v2.RandomHorizontalFlip(p=0.4),
      v2.ToDtype(torch.float16, scale=True),
      v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [14]:
def transform_to_tensor(transform, d): # d: a list of the PIL images
    images=[transform(img) for img in d]
    return images

# transform_to_tensor(transform, images)

In [21]:
batch=torch.stack(train_images)

TypeError: expected Tensor as element 0 in argument 0, but got JpegImageFile

### Transform via numpy

In [5]:
def transform_to_tensor_through_numpy(d):
    array=np.array(d)
    tensor=torch.from_numpy(array)

    return tensor

In [6]:
tensor_images=transform_to_tensor_through_numpy(train_images)

In [12]:
tensor_images

tensor([[[[114, 112, 113],
          [119, 117, 118],
          [117, 117, 115],
          ...,
          [124, 120, 119],
          [106, 102, 101],
          [ 54,  50,  49]],

         [[127, 125, 126],
          [127, 125, 126],
          [124, 124, 122],
          ...,
          [103,  99,  98],
          [ 81,  77,  76],
          [ 46,  42,  39]],

         [[133, 131, 132],
          [128, 126, 127],
          [125, 125, 125],
          ...,
          [ 72,  68,  65],
          [ 50,  46,  43],
          [ 40,  37,  32]],

         ...,

         [[ 36,  27,  20],
          [ 37,  28,  19],
          [ 39,  30,  21],
          ...,
          [ 89,  82,  74],
          [ 88,  81,  73],
          [ 89,  82,  74]],

         [[ 36,  29,  23],
          [ 38,  31,  23],
          [ 40,  33,  25],
          ...,
          [ 93,  84,  79],
          [ 88,  81,  75],
          [ 87,  80,  74]],

         [[ 35,  30,  24],
          [ 37,  32,  26],
          [ 42,  35,  27],
         